# Day 03：权重与偏置 —— 赋予神经元灵魂

> ❄️ 第一周 · AI 的初春与寒冬 · 第 3 天

昨天我们搭好了感知机的骨架——它知道自己要接收几个输入，但还不会思考。

今天我们给它注入灵魂：**权重（Weight）** 和 **偏置（Bias）**。

**今天的任务**：
1. 理解权重和偏置的直觉含义
2. 实现感知机的前向传播（Forward Pass）
3. 让感知机真正能做决策

---

## 1. 权重：每个输入有多重要？

还是用「今天去不去打球」的例子。你做决定时，天气和朋友叫不叫你，**重要程度完全不同**：

- 如果你是个超级篮球迷：天气的权重很高（5.0），朋友的权重较低（1.0）
- 如果你是个社牛：朋友叫你权重很高（5.0），天气无所谓（1.0）

**权重 = 你对某个因素的重视程度**

在神经网络中，**「学习」的本质就是不断调整这些权重**，直到找出最佳组合。

In [1]:
import torch

# 两种不同性格的人

# 篮球迷：天气最重要
basketball_fan_weights = torch.tensor([[5.0],   # 天气权重
                                        [1.0]])  # 朋友权重
print(f"篮球迷的权重: {basketball_fan_weights.T}")

# 社牛：朋友最重要
social_butterfly_weights = torch.tensor([[1.0],  # 天气权重
                                           [5.0]]) # 朋友权重
print(f"社牛的权重: {social_butterfly_weights.T}")

# 同样的输入，不同的人做出不同的决定 —— 因为权重不同！
situation = torch.tensor([[1.0, 0.0]])  # 晴天，朋友没叫

fan_score = torch.matmul(situation, basketball_fan_weights).item()
social_score = torch.matmul(situation, social_butterfly_weights).item()

print(f"\n情境: 晴天，朋友没叫")
print(f"篮球迷的得分: {fan_score}  (天气晴朗给了大分！)")
print(f"社牛的得分:   {social_score}  (没人叫，提不起劲...")

篮球迷的权重: tensor([[5., 1.]])
社牛的权重: tensor([[1., 5.]])

情境: 晴天，朋友没叫
篮球迷的得分: 5.0  (天气晴朗给了大分！)
社牛的得分:   1.0  (没人叫，提不起劲...


---

## 2. 偏置：你有多容易被说服？

假设你是一个**极度死宅**的人。哪怕天气再好、朋友再怎么叫，你都不太想出门。

这种「天生倾向」就是**偏置（Bias）**：
- 偏置为**很大的负数**：极其高冷，外界信号要非常强才能激活你
- 偏置为**0**：完全中性，只看外界输入
- 偏置为**正数**：非常外向，外界稍微给点信号你就兴奋

偏置独立于输入——它是你**与生俱来的「门槛」**。

In [2]:
import torch
# 三种不同性格的人，面对「晴天但朋友没叫」的情境
situation = torch.tensor([[1.0, 0.0]])  # 晴天，朋友没叫
weights = torch.tensor([[5.0], [1.0]])  # 相同的权重

# 基础分（不考虑偏置）
base_score = torch.matmul(situation, weights).item()
print(f"基础得分（不含偏置）: {base_score}")

# 三种性格
biases = {
    "死宅 (bias=-10.0)": -10.0,
    "普通人 (bias=-4.0)": -4.0,
    "社牛 (bias=+2.0)":   2.0,
}

print()
for label, bias in biases.items():
    final_score = base_score + bias
    decision = "去！" if final_score > 0 else "不去"
    print(f"{label:20s} -> 最终得分: {final_score:+.1f} -> {decision}")

# 偏置越大（越负），就越难被说服出门

基础得分（不含偏置）: 5.0

死宅 (bias=-10.0)      -> 最终得分: -5.0 -> 不去
普通人 (bias=-4.0)      -> 最终得分: +1.0 -> 去！
社牛 (bias=+2.0)       -> 最终得分: +7.0 -> 去！


---

## 3. 完整的感知机：前向传播

现在我们有了所有零件，组装一个完整的感知机！

前向传播（Forward Pass）的完整流程：

```text
输入 X ──→ 加权求和 (X·W + b) ──→ 激活函数 ──→ 输出
              ↑                       ↑
          权重W, 偏置b            阶跃函数
         (感知机的「价值观」)       (做最终决定)
```

In [3]:
import torch
# 本notebook独立定义完整的Perceptron类（含权重、偏置、前向传播）
class Perceptron:
    """
    完整的感知机 —— 可以接收输入、做加权求和、并输出决策
    """
    def __init__(self, input_features_count):
        self.input_features_count = input_features_count
        
        # 权重：随机初始化（模型是一张白纸）
        self.weights = torch.randn(input_features_count, 1)
        
        # 偏置：随机初始化
        self.bias = torch.randn(1)
        
    def forward(self, inputs):
        """
        前向传播：从输入到输出的完整计算过程
        """
        # 第一步：加权求和 Z = X·W + b
        z = torch.matmul(inputs, self.weights) + self.bias
        
        # 第二步：阶跃激活函数
        # 大于 0 -> 输出 1（激活/兴奋）
        # 小于等于 0 -> 输出 0（沉默）
        output = (z > 0).float()
        
        return output

print("感知机创建完成！")

感知机创建完成！


---

## 4. 实战测试：手动赋予「价值观」

我们先手动给感知机设定一组明确的权重，看看它会做出怎样的决策。

In [4]:
import torch

class Perceptron:
    """完整的感知机（含权重、偏置、前向传播）"""
    def __init__(self, input_features_count=2):
        self.weights = torch.randn(input_features_count, 1)
        self.bias = torch.randn(1)
    def forward(self, inputs):
        z = torch.matmul(inputs, self.weights) + self.bias
        return (z > 0).float()

import torch
# 创建感知机，手动设定权重（篮球迷版本）
neuron = Perceptron(input_features_count=2)
neuron.weights = torch.tensor([[5.0],   # 天气权重很大
                                 [1.0]])  # 朋友权重较小
neuron.bias = torch.tensor([-4.0])       # 偏置为负，比较宅

print(f"感知机的「价值观」:")
print(f"  天气权重: {neuron.weights[0].item()}")
print(f"  朋友权重: {neuron.weights[1].item()}")
print(f"  门槛(偏置): {neuron.bias.item()}")

# 测试四种情境
test_cases = [
    ("雨天，朋友没叫", [0.0, 0.0]),
    ("雨天，朋友叫了", [0.0, 1.0]),
    ("晴天，朋友没叫", [1.0, 0.0]),
    ("晴天，朋友叫了", [1.0, 1.0]),
]

print(f"\n{'情境':<16} {'输入':<16} {'得分':<10} {'决定'}")
print("-" * 55)

for description, values in test_cases:
    x = torch.tensor([values])
    z = torch.matmul(x, neuron.weights) + neuron.bias
    score = z.item()
    decision = neuron.forward(x).item()
    emoji = "去打球" if decision == 1 else "不去"
    print(f"{description:<16} {str(values):<16} {score:<+10.1f} {emoji}")

感知机的「价值观」:
  天气权重: 5.0
  朋友权重: 1.0
  门槛(偏置): -4.0

情境               输入               得分         决定
-------------------------------------------------------
雨天，朋友没叫          [0.0, 0.0]       -4.0       不去
雨天，朋友叫了          [0.0, 1.0]       -3.0       不去
晴天，朋友没叫          [1.0, 0.0]       +1.0       去打球
晴天，朋友叫了          [1.0, 1.0]       +2.0       去打球


### 解读

看看这个篮球迷的决策逻辑：
- **雨天+朋友叫**: 0x5 + 1x1 - 4 = **-3** -> 不去（雨天打败一切）
- **晴天+朋友没叫**: 1x5 + 0x1 - 4 = **+1** -> 去！（光靠晴天就够了）
- **晴天+朋友叫**: 1x5 + 1x1 - 4 = **+2** -> 去！（完美日）

权重和偏置的不同组合，决定了感知机截然不同的「性格」。

---

## 5. 小实验：改变权重，改变性格

试试给同一个感知机不同的权重，看它的行为如何变化。

In [5]:
import torch

class Perceptron:
    def __init__(self, input_features_count=2):
        self.weights = torch.randn(input_features_count, 1)
        self.bias = torch.randn(1)
    def forward(self, inputs):
        z = torch.matmul(inputs, self.weights) + self.bias
        return (z > 0).float()

neuron = Perceptron(input_features_count=2)

import torch
# 场景：晴天但朋友没叫
situation = torch.tensor([[1.0, 0.0]])

personalities = [
    ("死宅",    torch.tensor([[-2.0], [-2.0]]), torch.tensor([-1.0])),
    ("篮球迷",  torch.tensor([[ 5.0], [ 1.0]]), torch.tensor([-4.0])),
    ("社牛",    torch.tensor([[ 1.0], [ 5.0]]), torch.tensor([-2.0])),
    ("无脑冲",  torch.tensor([[ 3.0], [ 3.0]]), torch.tensor([ 2.0])),
]

print(f"情境: 晴天，朋友没叫 [1, 0]\n")

for name, w, b in personalities:
    neuron.weights = w
    neuron.bias = b
    score = torch.matmul(situation, w).item() + b.item()
    result = neuron.forward(situation).item()
    print(f"{name:6s}: 得分 {score:+.1f} -> {'去！' if result == 1 else '不去'}")

print(f"\n同一个输入，不同的「价值观(权重)」，得出完全不同的结论。")
print(f"这就是为什么神经网络需要「学习」——找到最佳的权重组合。")

情境: 晴天，朋友没叫 [1, 0]

死宅    : 得分 -3.0 -> 不去
篮球迷   : 得分 +1.0 -> 去！
社牛    : 得分 -1.0 -> 不去
无脑冲   : 得分 +5.0 -> 去！

同一个输入，不同的「价值观(权重)」，得出完全不同的结论。
这就是为什么神经网络需要「学习」——找到最佳的权重组合。


---

## 今日总结

| 概念 | 直觉理解 | 数学表达 |
|------|---------|----------|
| 权重 W | 每个输入的重要程度 | 每个特征的乘数 |
| 偏置 b | 激活的基本门槛 | 独立于输入的常数 |
| 前向传播 | 从输入到输出的计算过程 | Z = X·W + b -> 阶跃函数 |
| 「学习」 | 找到最佳的 W 和 b | 调整参数使输出匹配期望 |

**明天预告**：用可视化的方式，在二维坐标系中画出感知机的「决策边界」——那条它试图画出的「直线」。你会直观地看到，为什么它能解决 AND/OR，却对 XOR 束手无策。